# Ordered Logistic Regression Results Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset is provided via a Croissant schema URL. The dataset includes ordered logistic regression outputs, coefficients, standard errors, and p-values for variables associated with household adoption of indigenous and modern knowledge in rangeland management interventions across Northern Kenya.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Display dataset name and description
md = dataset.metadata
print(f"{md.name}: {md.description}")

## 2. Data Overview
Review available record sets and their IDs as defined by the schema. You can use the `.record_sets` property to list all record sets and inspect their fields and columns using their `@id`s.

In [ ]:
# List all available record sets with their @id and @type
print("Available record sets in this dataset:")
for record_set in dataset.record_sets:
    print(f"@id: {record_set['@id']}, @type: {record_set.get('@type', 'n/a')}, name: {record_set.get('name', 'n/a')}")
    if 'field' in record_set:
        fields = record_set['field']
        if isinstance(fields, dict):
            fields = [fields]
        for field in fields:
            print(f"    Field @id: {field['@id']}, name: {field.get('name', 'n/a')}")

## 3. Data Extraction
Load data from one or more record sets into pandas DataFrames. All record sets, fields, and columns are referenced by their `@id`. First, we create a list of the record set `@id`s and display available columns for initial exploration.

In [ ]:
# Extract data from each record set into DataFrames, using @id references
# Get the list of record set @id's
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records from record set '{record_set_id}'")

# For demonstration, print available columns from the first record set with data
if record_set_ids:
    first_id = record_set_ids[0]
    print(f"\nFields for record set '{first_id}':\n", dataframes[first_id].columns.tolist())
    display(dataframes[first_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria (using `@id` for fields), normalizing numeric fields, and grouping data by key attributes.

In [ ]:
from IPython.display import display

# For illustration, choose a numeric field's @id. Replace this with a valid @id from your dataset fields.
# Here, we try likely candidates such as coefficient, log_likelihood, or similar statistical measure columns.
default_record_set_id = record_set_ids[0] if record_set_ids else None
df = dataframes[default_record_set_id] if default_record_set_id else pd.DataFrame()

print("Available fields:", df.columns.tolist())

# Attempt to heuristically pick a numeric field @id
import re
numeric_field_candidates = [col for col in df.columns if re.search(r'(coefficient|log_likelihood|p_value|std_error|beta|estimate|value)', col, re.IGNORECASE)]
numeric_field = numeric_field_candidates[0] if numeric_field_candidates else None
if not numeric_field:
    numeric_field = df.select_dtypes('number').columns[0] if not df.select_dtypes('number').empty else df.columns[0] if len(df.columns) else None

if numeric_field:
    # Set a threshold value. We'll use the mean if min/max are not known.
    threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())
    # Normalize the numeric field
    col_norm = f"{numeric_field}_normalized"
    filtered_df[col_norm] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, col_norm]].head())
    # Group by a string/categorical field, if available
    group_field_candidates = [col for col in df.columns if col != numeric_field and df[col].dtype == object]
    group_field = group_field_candidates[0] if group_field_candidates else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field, as_index=False)[numeric_field].mean()
        print(f"Grouped data by {group_field} (mean of {numeric_field}):")
        display(grouped_df.head())
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field, and, if available, visualize grouping by a categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field and not df.empty:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of '{numeric_field}'")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()

    # If we have a group field, show aggregated barplot
    if group_field:
        plt.figure(figsize=(8, 4))
        order = grouped_df.sort_values(numeric_field, ascending=False)[group_field] if not grouped_df.empty else None
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field, order=order)
        plt.xticks(rotation=45)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.tight_layout()
        plt.show()
else:
    print("Cannot visualize: no suitable data found.")

## 6. Conclusion
In this notebook, we've loaded the dataset defined by a Croissant schema and explored its structure and content using `mlcroissant`. We examined record sets using their `@id`s, selected numeric fields for basic data processing, filtered and normalized results, grouped data for further insight, and visualized key data distributions. For more advanced analysis, tailor the code with specific field `@id`s found in your dataset overview.